In [0]:
%run ../config/utils

In [0]:
import pyspark.sql.functions as f
from pyspark.sql import DataFrame
from functools import reduce
import matplotlib.pyplot as plt


In [0]:
def normalize_like(df_to_fix: DataFrame, df_ref: DataFrame) -> DataFrame:
    """
    Cast df_to_fix columns to match df_ref schema, round decimals/doubles to 2 decimal places,
    and order by all columns.
    Assumes df_ref has the correct schema.
    """
    # Get columns and types from the reference dataframe
    ref_schema = df_ref.schema
    ref_cols = [field.name for field in ref_schema]
    ref_types = {field.name: field.dataType.simpleString() for field in ref_schema}

    # Ensure all reference columns exist in df_to_fix
    df_fixed = df_to_fix
    for col in ref_cols:
        if col not in df_fixed.columns:
            df_fixed = df_fixed.withColumn(col, f.lit(None).cast(ref_types[col]))

    # Select in the same order, cast types, and round decimals/doubles
    select_exprs = []
    for col in ref_cols:
        col_type = ref_types[col]
        col_expr = f.col(col).cast(col_type)
        
        if col_type.startswith("decimal") or col_type == "double":
            col_expr = col_expr # round to 2 decimals
        
        select_exprs.append(col_expr.alias(col))

    df_fixed = df_fixed.select(*select_exprs)

    # Order by all columns for deterministic comparison
    df_fixed = df_fixed.orderBy(*ref_cols)

    return df_fixed

def diff_report(df1, df2, keys):
    l, r = df1.alias("l"), df2.alias("r")
    key_cond = reduce(lambda a,b: a & b, [f.col(f"l.{k}").eqNullSafe(f.col(f"r.{k}")) for k in keys])
    joined = l.join(r, on=key_cond, how="inner")

    non_keys = [c for c in df1.columns if c not in keys]
    diffs = [
        f.when(~f.col(f"l.{c}").eqNullSafe(f.col(f"r.{c}")),
               f.struct(f.col(f"l.{c}").alias("left"), f.col(f"r.{c}").alias("right"))
        ).alias(c)
        for c in non_keys
    ]
    out = joined.select(*[f.col(f"l.{k}").alias(k) for k in keys], *diffs)
    cond_any = reduce(lambda a,b: a | b, [f.col(c).isNotNull() for c in non_keys]) if non_keys else f.lit(False)
    return out.filter(cond_any)


## Trip Spend models comparison

In [0]:
df_s3 = spark.read.csv("s3://memberanalytics-data-out-prod/MODELDATA/PREDICTIONS/TRIP_SPEND_MODELS/prod_2025_11_16/trip_spend_predictions_2025-11-15.csv", header=True, inferSchema=True).dropna(subset=["MBRSHP_SID"])
df_dbx = spark.table(trip_spend_prediction).filter("FISCAL_WEEK_END='2025-11-15'").drop('FISCAL_WEEK_END', 'predicted_spend')

In [0]:
if df_s3.count() != df_dbx.count(): print(f"Warning: row counts differ: df_s3={df_s3.count()}, df_dbx={df_dbx.count()}") 
else: print("Row count matches")

In [0]:
df_dbx_norm = normalize_like(df_dbx, df_dbx)
df_s3_norm = normalize_like(df_s3, df_dbx)

In [0]:
df1 = df_dbx_norm.toPandas()
df2 = df_s3_norm.toPandas()
plt.figure(figsize=(10,6))
# df1['probability_making_a_trip'] = round(df1['probability_making_a_trip'],2)
# df2['probability_making_a_trip'] = round(df2['probability_making_a_trip'],2)
df1['probability_making_a_trip'].plot.hist(bins=100, alpha=0.5, label='Table Databricks')
df2['probability_making_a_trip'].plot.hist(bins=100, alpha=0.5, label='Table S3')
plt.xlabel('Score')
plt.ylabel('Frequency')
plt.title('Score Distribution Comparison')
plt.legend()
plt.show()

In [0]:
# df_dbx_norm = df_dbx_norm.withColumn("probability_making_a_trip", f.round(f.col("probability_making_a_trip"), 1))
# df_s3_norm = df_s3_norm.withColumn("probability_making_a_trip", f.round(f.col("probability_making_a_trip"), 1))

In [0]:
diff = diff_report(df_dbx_norm, df_s3_norm, ['MBRSHP_SID'])
diff.display()

In [0]:
df_calculated = diff.withColumn(
    "diff_value", 
    f.abs(f.col("probability_making_a_trip")["left"] - f.col("probability_making_a_trip")["right"])
)
df_calculated.filter(f.col("diff_value") > 0.1).display()

## BBM propensity model comparison

In [0]:
df_s3 = spark.read.csv("s3://memberanalytics-data-out-prod/MODELDATA/BBM_PROPENSITY/BBM_SALES_SCORE/MKT_EDW_BBM_SCORE_20251108.csv", header=True, inferSchema=True)#.dropna(subset=["MBRSHP_SID"])
df_dbx = spark.table('datascience_ea_dev.pe.bbm_inference_output').filter("run_name='2025-11-08'").drop('run_name', 'run_date').dropDuplicates(subset=["mbrshp_nbr"])

In [0]:
if df_s3.count() != df_dbx.count(): print(f"Warning: row counts differ: df_s3={df_s3.count()}, df_dbx={df_dbx.count()}") 
else: print("Row count matches")

In [0]:
df_dbx_norm = normalize_like(df_dbx, df_dbx)
df_s3_norm = normalize_like(df_s3, df_dbx)

In [0]:
df1 = df_dbx_norm.toPandas()
df2 = df_s3_norm.toPandas()
plt.figure(figsize=(10,6))
# df1['probability_making_a_trip'] = round(df1['probability_making_a_trip'],2)
# df2['probability_making_a_trip'] = round(df2['probability_making_a_trip'],2)
df1['score'].plot.hist(bins=50, alpha=0.5, label='Table Databricks')
df2['score'].plot.hist(bins=50, alpha=0.5, label='Table S3')
plt.xlabel('Score')
plt.ylabel('Frequency')
plt.title('Score Distribution Comparison')
plt.legend()
plt.show()

In [0]:
df_dbx_norm = df_dbx_norm.withColumn("score", f.round(f.col("score"), 1))
df_s3_norm = df_s3_norm.withColumn("score", f.round(f.col("score"), 1))

In [0]:
diff = diff_report(df_dbx_norm, df_s3_norm, ['mbrshp_nbr'])
diff.display()

In [0]:
df_calculated = diff.withColumn(
    "diff_value", 
    f.abs(f.col("score")["left"] - f.col("score")["right"])
)
df_calculated.filter(f.col("diff_value") > 0.2).display()

## Digital propensity Sales model comparison

In [0]:
df_s3 = spark.read.csv("s3://memberanalytics-data-out-prod/MODELDATA/Digital_propensity_scoring/sales/sales_scored_2025-11-08.csv", header=True, inferSchema=True)#.dropna(subset=["MBRSHP_SID"])
df_dbx = spark.table(digital_sales_scores).filter("FISCAL_WEEK_END='2025-11-08'")#.drop('run_name', 'run_date').dropDuplicates(subset=["mbrshp_nbr"])

In [0]:
if df_s3.count() != df_dbx.count(): print(f"Warning: row counts differ: df_s3={df_s3.count()}, df_dbx={df_dbx.count()}") 
else: print("Row count matches")

In [0]:
df_dbx_norm = normalize_like(df_dbx, df_dbx)
df_s3_norm = normalize_like(df_s3, df_dbx)

In [0]:
df1 = df_dbx_norm.toPandas()
df2 = df_s3_norm.toPandas()
plt.figure(figsize=(10,6))
# df1['probability_making_a_trip'] = round(df1['probability_making_a_trip'],2)
# df2['probability_making_a_trip'] = round(df2['probability_making_a_trip'],2)
df1['sales_score'].plot.hist(bins=50, alpha=0.5, label='Table Databricks')
df2['sales_score'].plot.hist(bins=50, alpha=0.5, label='Table S3')
plt.xlabel('Score')
plt.ylabel('Frequency')
plt.title('Score Distribution Comparison')
plt.legend()
plt.show()

In [0]:
df_dbx_norm = df_dbx_norm.withColumn("sales_score", f.round(f.col("sales_score"), 1)).drop('LATEST_MBRSHP_NBR')
df_s3_norm = df_s3_norm.withColumn("sales_score", f.round(f.col("sales_score"), 1)).drop('LATEST_MBRSHP_NBR')

In [0]:
diff = diff_report(df_dbx_norm, df_s3_norm, ['MBRSHP_SID'])
diff.display()

In [0]:
df_calculated = diff.withColumn(
    "diff_value", 
    f.abs(f.col("sales_score")["left"] - f.col("sales_score")["right"])
)
df_calculated.filter(f.col("diff_value") > 0.2).display()

## Digital propensity Trips model comparison

In [0]:
df_s3 = spark.read.csv("s3://memberanalytics-data-out-prod/MODELDATA/Digital_propensity_scoring/trips/trips_scored_2025-11-08.csv", header=True, inferSchema=True)#.dropna(subset=["MBRSHP_SID"])
df_dbx = spark.table(digital_trips_scores).filter("FISCAL_WEEK_END='2025-11-08'")#.drop('run_name', 'run_date').dropDuplicates(subset=["mbrshp_nbr"])

In [0]:
if df_s3.count() != df_dbx.count(): print(f"Warning: row counts differ: df_s3={df_s3.count()}, df_dbx={df_dbx.count()}") 
else: print("Row count matches")

In [0]:
df_dbx_norm = normalize_like(df_dbx, df_dbx)
df_s3_norm = normalize_like(df_s3, df_dbx)

In [0]:
df1 = df_dbx_norm.toPandas()
df2 = df_s3_norm.toPandas()
plt.figure(figsize=(10,6))
# df1['probability_making_a_trip'] = round(df1['probability_making_a_trip'],2)
# df2['probability_making_a_trip'] = round(df2['probability_making_a_trip'],2)
df1['trips_score'].plot.hist(bins=50, alpha=0.5, label='Table Databricks')
df2['trips_score'].plot.hist(bins=50, alpha=0.5, label='Table S3')
plt.xlabel('Score')
plt.ylabel('Frequency')
plt.title('Score Distribution Comparison')
plt.legend()
plt.show()

In [0]:
df_dbx_norm = df_dbx_norm.withColumn("trips_score", f.round(f.col("trips_score"), 1)).drop('LATEST_MBRSHP_NBR')
df_s3_norm = df_s3_norm.withColumn("trips_score", f.round(f.col("trips_score"), 1)).drop('LATEST_MBRSHP_NBR')

In [0]:
diff = diff_report(df_dbx_norm, df_s3_norm, ['MBRSHP_SID'])
diff.display()

In [0]:
df_calculated = diff.withColumn(
    "diff_value", 
    f.abs(f.col("trips_score")["left"] - f.col("trips_score")["right"])
)
df_calculated.filter(f.col("diff_value") > 0.2).display()

In [0]:
df_s3.filter("MBRSHP_SID is NULL").display()

## CF model comparison

In [0]:
df_s3 = spark.read.parquet("s3://memberanalytics-data-out-prod/MODELDATA/PREDICTIONS/CF_MODEL/prod_2025_11_23/20241122-20251122-combined/preds-20251127-combined/PARQUET")#.dropna(subset=["MBRSHP_SID"])
df_dbx = spark.table(cf_prediction).filter("END_DATE='2025-11-22'").drop('CATEGORY_CD', 'START_DATE','END_DATE','RUN_NAME')

In [0]:
if df_s3.count() != df_dbx.count(): print(f"Warning: row counts differ: df_s3={df_s3.count()}, df_dbx={df_dbx.count()}") 
else: print("Row count matches")

In [0]:
df_dbx_norm = normalize_like(df_dbx, df_dbx).limit(100000)
df_s3_norm = normalize_like(df_s3, df_dbx).join(df_dbx_norm.select('MBRSHP_SID','CATEGORY_ID'), ['MBRSHP_SID','CATEGORY_ID'], 'inner')

In [0]:
df1 = df_dbx_norm.toPandas()
df2 = df_s3_norm.toPandas()
plt.figure(figsize=(10,6))
# df1['probability_making_a_trip'] = round(df1['probability_making_a_trip'],2)
# df2['probability_making_a_trip'] = round(df2['probability_making_a_trip'],2)
df1['prediction'].plot.hist(bins=50, alpha=0.5, label='Table Databricks')
df2['prediction'].plot.hist(bins=50, alpha=0.5, label='Table S3')
plt.xlabel('Score')
plt.ylabel('Frequency')
plt.title('Score Distribution Comparison')
plt.legend()
plt.show()

In [0]:
df_dbx_norm = normalize_like(df_dbx, df_dbx)
df_s3_norm = normalize_like(df_s3, df_dbx)

In [0]:
df_dbx_norm = df_dbx_norm.withColumn("prediction", f.round(f.col("prediction"), 1)).select('MBRSHP_SID', 'CATEGORY_ID','prediction')
df_s3_norm = df_s3_norm.withColumn("prediction", f.round(f.col("prediction"), 1)).select('MBRSHP_SID', 'CATEGORY_ID','prediction')

In [0]:
diff = diff_report(df_dbx_norm, df_s3_norm, ['MBRSHP_SID', 'CATEGORY_ID'])
diff.display()

In [0]:
df_calculated = diff.withColumn(
    "diff_value", 
    f.abs(f.col("prediction")["left"] - f.col("prediction")["right"])
)
df_calculated.filter(f.col("diff_value") > 0.2).display()

## GM model comparison

In [0]:
df_s3 = spark.read.csv("s3://memberanalytics-data-out-prod/USERS/skarunanithi/GM_Propensity_Model/2025/GM_SCORE_20250922.csv", header=True, inferSchema=True)#.dropna(subset=["MBRSHP_SID"])
df_dbx = spark.table(gm_scores).filter("score_date='2025-10-30'").withColumn('score',f.col('score').cast('double'))#.drop('CATEGORY_CD', 'START_DATE','END_DATE','RUN_NAME')

In [0]:
if df_s3.count() != df_dbx.count(): print(f"Warning: row counts differ: df_s3={df_s3.count()}, df_dbx={df_dbx.count()}") 
else: print("Row count matches")

In [0]:
df_dbx_norm = normalize_like(df_dbx, df_dbx)
df_s3_norm = normalize_like(df_s3, df_dbx)

In [0]:
df1 = df_dbx_norm.toPandas()
df2 = df_s3_norm.toPandas()
plt.figure(figsize=(10,6))
# df1['probability_making_a_trip'] = round(df1['probability_making_a_trip'],2)
# df2['probability_making_a_trip'] = round(df2['probability_making_a_trip'],2)
df1['score'].plot.hist(bins=50, alpha=0.5, label='Table Databricks')
df2['score'].plot.hist(bins=50, alpha=0.5, label='Table S3')
plt.xlabel('Score')
plt.ylabel('Frequency')
plt.title('Score Distribution Comparison')
plt.legend()
plt.show()

In [0]:
df_dbx_norm = df_dbx_norm.withColumn("score", f.round(f.col("score"), 1)).drop('score_date','mbrshp_nbr')
df_s3_norm = df_s3_norm.withColumn("score", f.round(f.col("score"), 1)).drop('score_date','mbrshp_nbr')

In [0]:
diff = diff_report(df_dbx_norm, df_s3_norm, ['MBRSHP_SID'])
diff.display()

In [0]:
df_calculated = diff.withColumn(
    "diff_value", 
    f.abs(f.col("score")["left"] - f.col("score")["right"])
)
df_calculated.filter(f.col("diff_value") > 0.41).display()